In [3]:
import sympy as sp

In [4]:
from functools import lru_cache

@lru_cache(maxsize = None)
def get_ckn(k: int, n: int, p):
    if k<0 or k>n:
        return sp.Integer(0)
    if n == 0:
        return sp.Integer(1) if k==0 else sp.Integer(0)
    return get_ckn(k-1, n-1, p)/(2*p) + (k+1)*get_ckn(k+1, n-1, p)


### Implementation of overlap integrals

In [5]:
alpha, beta = sp.symbols('alpha beta', real = True, positive = True)
Ax, Ay, Az = sp.symbols('Ax Ay Az', real = True)
Bx, By, Bz = sp.symbols('Bx By Bz', real = True)

p = alpha + beta
q = alpha*beta/p
RAB2 = (Ax-Bx)**2 + (Ay-By)**2 + (Az-Bz)**2


In [6]:
S00 = (sp.sqrt(sp.pi/p))**3 * sp.exp(-q*RAB2)
S00

pi**(3/2)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(3/2)

In [7]:
def build_derivative_table(base_integral, Lmax, Avars, Bvars, indices):
    Ax, Ay, Az = Avars
    Bx, By, Bz = Bvars
    
    @lru_cache(maxsize=None)
    def derivative(i,j,k,l,m,n):
        if (i,j,k,l,m,n) == (0,0,0,0,0,0):
            return base_integral
        if i>0:
            return sp.diff(derivative(i-1,j,k,l,m,n),Ax)
        if j>0:
            return sp.diff(derivative(i,j-1,k,l,m,n),Ay)
        if k>0:
            return sp.diff(derivative(i,j,k-1,l,m,n),Az) 
        if l>0:
            return sp.diff(derivative(i,j,k,l-1,m,n),Bx)
        if m>0:
            return sp.diff(derivative(i,j,k,l,m-1,n),By)
        return sp.diff(derivative(i,j,k,l,m,n-1),Bz) 
    
    derivatives_dict = {}
    for (i,j,k,l,m,n) in indices:
        derivatives_dict[(i,j,k,l,m,n)] = derivative(i,j,k,l,m,n).simplify()
    return derivatives_dict

## Generate all induces up to Lmax

In [8]:
Lmax = 2

In [ ]:
def l_to_ijk(L):
    return [
        (i, j, L - i - j)
        for i in range(L, -1, -1)
        for j in range(L - i, -1, -1)
    ]
integral_indices=[]
ijk = [t for L in range(Lmax+1) for t in l_to_ijk(L)]
for i in ijk:
    for j in ijk:
        integral_indices.append(i+j)
integral_indices

[(0, 0, 0, 0, 0, 0),
 (0, 0, 0, 1, 0, 0),
 (0, 0, 0, 0, 1, 0),
 (0, 0, 0, 0, 0, 1),
 (0, 0, 0, 2, 0, 0),
 (0, 0, 0, 1, 1, 0),
 (0, 0, 0, 1, 0, 1),
 (0, 0, 0, 0, 2, 0),
 (0, 0, 0, 0, 1, 1),
 (0, 0, 0, 0, 0, 2),
 (1, 0, 0, 0, 0, 0),
 (1, 0, 0, 1, 0, 0),
 (1, 0, 0, 0, 1, 0),
 (1, 0, 0, 0, 0, 1),
 (1, 0, 0, 2, 0, 0),
 (1, 0, 0, 1, 1, 0),
 (1, 0, 0, 1, 0, 1),
 (1, 0, 0, 0, 2, 0),
 (1, 0, 0, 0, 1, 1),
 (1, 0, 0, 0, 0, 2),
 (0, 1, 0, 0, 0, 0),
 (0, 1, 0, 1, 0, 0),
 (0, 1, 0, 0, 1, 0),
 (0, 1, 0, 0, 0, 1),
 (0, 1, 0, 2, 0, 0),
 (0, 1, 0, 1, 1, 0),
 (0, 1, 0, 1, 0, 1),
 (0, 1, 0, 0, 2, 0),
 (0, 1, 0, 0, 1, 1),
 (0, 1, 0, 0, 0, 2),
 (0, 0, 1, 0, 0, 0),
 (0, 0, 1, 1, 0, 0),
 (0, 0, 1, 0, 1, 0),
 (0, 0, 1, 0, 0, 1),
 (0, 0, 1, 2, 0, 0),
 (0, 0, 1, 1, 1, 0),
 (0, 0, 1, 1, 0, 1),
 (0, 0, 1, 0, 2, 0),
 (0, 0, 1, 0, 1, 1),
 (0, 0, 1, 0, 0, 2),
 (2, 0, 0, 0, 0, 0),
 (2, 0, 0, 1, 0, 0),
 (2, 0, 0, 0, 1, 0),
 (2, 0, 0, 0, 0, 1),
 (2, 0, 0, 2, 0, 0),
 (2, 0, 0, 1, 1, 0),
 (2, 0, 0, 1, 0, 1),
 (2, 0, 0, 0,

In [10]:
derivatives_dict = build_derivative_table(S00, Lmax, (Ax,Ay,Az), (Bx,By,Bz), integral_indices)

## Generate only canonical indices

In [11]:
from itertools import permutations

S3 = list(permutations((0,1,2)))

def permute(ijk, permutation):
    return tuple(ijk[i] for i in permutation)

def l_to_ijk(L):
    return [(i, j, L - i - j) for i in range(L + 1) for j in range(L+1-i)]

def get_canonical_indices(La, Lb):
    canonical_indices = set()
    for (i,j,k) in l_to_ijk(La):
        for (l,m,n) in l_to_ijk(Lb):
            orbit_ijklmn = set()
            for permutation in S3:
                permuted_ijk = permute((i,j,k), permutation)
                permuted_lmn = permute((l,m,n), permutation)
                idx = permuted_ijk + permuted_lmn
                orbit_ijklmn.add(idx)
                idx = permuted_lmn + permuted_ijk
                orbit_ijklmn.add(idx)
            canonical_indices.add(min(orbit_ijklmn))
    return canonical_indices
                
indices = set()
for La in range(Lmax+1):
    for Lb in range(Lmax+1):
        indices.update(get_canonical_indices(La, Lb))   
indices = sorted(indices)
canonical_to_rep = {idx: num for num, idx in enumerate(indices)}
canonical_to_rep

{(0, 0, 0, 0, 0, 0): 0,
 (0, 0, 0, 0, 0, 1): 1,
 (0, 0, 0, 0, 0, 2): 2,
 (0, 0, 0, 0, 1, 1): 3,
 (0, 0, 1, 0, 0, 1): 4,
 (0, 0, 1, 0, 0, 2): 5,
 (0, 0, 1, 0, 1, 0): 6,
 (0, 0, 1, 0, 1, 1): 7,
 (0, 0, 1, 0, 2, 0): 8,
 (0, 0, 1, 1, 1, 0): 9,
 (0, 0, 2, 0, 0, 2): 10,
 (0, 0, 2, 0, 1, 1): 11,
 (0, 0, 2, 0, 2, 0): 12,
 (0, 0, 2, 1, 1, 0): 13,
 (0, 1, 1, 0, 1, 1): 14,
 (0, 1, 1, 1, 0, 1): 15}

### Generate permutation mappings
For each set of indices generate a permutation that maps the given set to a canonical index.
Construct the dictionary with key = ijklmn and values = (permutation, swap)

In [12]:
ijklmn_to_rep = {}

for La in range(Lmax+1):
    for Lb in range(Lmax+1):
        ijk_list = l_to_ijk(La)
        lmn_list = l_to_ijk(Lb)
        for ijk in ijk_list:
            for lmn in lmn_list:
                ijklmn = ijk + lmn
                for canonical in indices:
                    for perm in S3:
                        pijk = permute(ijk, perm)
                        plmn = permute(lmn, perm)
                        pijklmn = pijk + plmn
                        if pijklmn == canonical:
                            ijklmn_to_rep[ijklmn] = (canonical_to_rep[canonical],perm, False)
                        plmnijk = plmn + pijk
                        if plmnijk == canonical:
                            ijklmn_to_rep[ijklmn] = (canonical_to_rep[canonical],perm, True)



In [13]:
from pprint import pformat

print(f"IJKLMN_TO_REP = {pformat(ijklmn_to_rep, sort_dicts=False)}")

IJKLMN_TO_REP = {(0, 0, 0, 0, 0, 0): (0, (2, 1, 0), True),
 (0, 0, 0, 0, 0, 1): (1, (1, 0, 2), False),
 (0, 0, 0, 0, 1, 0): (1, (2, 0, 1), False),
 (0, 0, 0, 1, 0, 0): (1, (2, 1, 0), False),
 (0, 0, 0, 0, 0, 2): (2, (1, 0, 2), False),
 (0, 0, 0, 0, 1, 1): (3, (0, 2, 1), False),
 (0, 0, 0, 0, 2, 0): (2, (2, 0, 1), False),
 (0, 0, 0, 1, 0, 1): (3, (1, 2, 0), False),
 (0, 0, 0, 1, 1, 0): (3, (2, 1, 0), False),
 (0, 0, 0, 2, 0, 0): (2, (2, 1, 0), False),
 (0, 0, 1, 0, 0, 0): (1, (1, 0, 2), True),
 (0, 1, 0, 0, 0, 0): (1, (2, 0, 1), True),
 (1, 0, 0, 0, 0, 0): (1, (2, 1, 0), True),
 (0, 0, 1, 0, 0, 1): (4, (1, 0, 2), True),
 (0, 0, 1, 0, 1, 0): (6, (0, 2, 1), True),
 (0, 0, 1, 1, 0, 0): (6, (1, 2, 0), True),
 (0, 1, 0, 0, 0, 1): (6, (0, 2, 1), False),
 (0, 1, 0, 0, 1, 0): (4, (2, 0, 1), True),
 (0, 1, 0, 1, 0, 0): (6, (2, 1, 0), True),
 (1, 0, 0, 0, 0, 1): (6, (1, 2, 0), False),
 (1, 0, 0, 0, 1, 0): (6, (2, 1, 0), False),
 (1, 0, 0, 1, 0, 0): (4, (2, 1, 0), True),
 (0, 0, 1, 0, 0, 2): (5, (

In [57]:
LA_LB_TO_IJKLMN = {}
for la in range(Lmax+1):
    for lb in range(Lmax+1):
        ijklmn_list = []
        for ijk in l_to_ijk(la):
            for lmn in l_to_ijk(lb):
                ijklmn_list.append(ijk + lmn)
        LA_LB_TO_IJKLMN[(la,lb)]=ijklmn_list
print(f"LA_LB_TO_IJKLMN = {pformat(LA_LB_TO_IJKLMN, sort_dicts=False)}")

LA_LB_TO_IJKLMN = {(0, 0): [(0, 0, 0, 0, 0, 0)],
 (0, 1): [(0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0)],
 (0, 2): [(0, 0, 0, 0, 0, 2),
          (0, 0, 0, 0, 1, 1),
          (0, 0, 0, 0, 2, 0),
          (0, 0, 0, 1, 0, 1),
          (0, 0, 0, 1, 1, 0),
          (0, 0, 0, 2, 0, 0)],
 (1, 0): [(0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0)],
 (1, 1): [(0, 0, 1, 0, 0, 1),
          (0, 0, 1, 0, 1, 0),
          (0, 0, 1, 1, 0, 0),
          (0, 1, 0, 0, 0, 1),
          (0, 1, 0, 0, 1, 0),
          (0, 1, 0, 1, 0, 0),
          (1, 0, 0, 0, 0, 1),
          (1, 0, 0, 0, 1, 0),
          (1, 0, 0, 1, 0, 0)],
 (1, 2): [(0, 0, 1, 0, 0, 2),
          (0, 0, 1, 0, 1, 1),
          (0, 0, 1, 0, 2, 0),
          (0, 0, 1, 1, 0, 1),
          (0, 0, 1, 1, 1, 0),
          (0, 0, 1, 2, 0, 0),
          (0, 1, 0, 0, 0, 2),
          (0, 1, 0, 0, 1, 1),
          (0, 1, 0, 0, 2, 0),
          (0, 1, 0, 1, 0, 1),
          (0, 1, 0, 1, 1, 0),
          (0, 1, 0, 2, 0, 0

In [14]:
def get_integral_expressions(integral_indices, derivatives_dict, alpha, beta):
    integral_expressions = {}

    for (i,j,k,l,m,n) in integral_indices:

        ci = [get_ckn(o, i, alpha) for o in range(i + 1)]
        cj = [get_ckn(p, j, alpha) for p in range(j + 1)]
        ck = [get_ckn(q, k, alpha) for q in range(k + 1)]

        cl = [get_ckn(r, l, beta) for r in range(l + 1)]
        cm = [get_ckn(s, m, beta) for s in range(m + 1)]
        cn = [get_ckn(t, n, beta) for t in range(n + 1)]

        expr = 0
        for o, co in enumerate(ci):
            for p, cp in enumerate(cj):
                for q, cq in enumerate(ck):
                    for r, cr in enumerate(cl):
                        for s, cs in enumerate(cm):
                            for t, ct in enumerate(cn):
                                expr += (
                                    co * cp * cq * cr * cs * ct
                                    * derivatives_dict[(o, p, q, r, s, t)]
                                )

        integral_expressions[(i,j,k,l,m,n)] = sp.simplify(expr)
    return integral_expressions

In [15]:
integral_expressions = get_integral_expressions(indices, derivatives_dict, alpha, beta)

In [16]:
Dx, Dy, Dz, D2, P, Q = sp.symbols(
    'Dx Dy Dz D2 P Q',
    real=True,
)

subsdict = {
    Ax - Bx: Dx,
    Ay - By: Dy,
    Az - Bz: Dz,
    (Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2: RAB2,
    alpha + beta: P,
    alpha * beta: Q,
}
for key, value in integral_expressions.items():
    integral_expressions[key] = value.subs(subsdict)
    print(key, integral_expressions[key])

(0, 0, 0, 0, 0, 0) pi**(3/2)*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/P**(3/2)
(0, 0, 0, 0, 0, 1) pi**(3/2)*Dz*alpha*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/P**(5/2)
(0, 0, 0, 0, 0, 2) pi**(3/2)*(P**2 - alpha*(-2*Dz**2*Q + P))*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/(2*P**(7/2)*beta)
(0, 0, 0, 0, 1, 1) pi**(3/2)*Dy*Dz*alpha**2*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/P**(7/2)
(0, 0, 1, 0, 0, 1) pi**(3/2)*(-2*Dz**2*Q + P)*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/(2*P**(7/2))
(0, 0, 1, 0, 0, 2) pi**(3/2)*(-Dz*P**2 + Dz*alpha*(-2*Dz**2*Q + 3*alpha + 3*beta))*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/(2*P**(9/2))
(0, 0, 1, 0, 1, 0) -pi**(3/2)*Dy*Dz*Q*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/P**(7/2)
(0, 0, 1, 0, 1, 1) pi**(3/2)*Dy*alpha*(-2*Dz**2*Q + P)*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/(2*P**(9/2))
(0, 0, 1, 0, 2, 0) pi**(3/2)*(-Dz*P**2 - Dz*alpha*(2*Dy**2*Q - P))*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/(2*P**(9/2))
(0, 0, 1, 1, 1, 0) -pi**(3/2)*Dx*Dy*Dz*Q*alpha*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/P**(9/2)
(0, 0, 2, 0, 0, 2) pi**(3/2)*(P**4 - 

In [17]:
from sympy.printing.numpy import NumPyPrinter, _known_constants_numpy, _known_functions_numpy

printer = NumPyPrinter()
printer._module = "np"
printer.known_functions = { k:f"np.{v}" for k,v in _known_functions_numpy.items()}
printer.known_constants = {k: f"np.{v}" for k, v in _known_constants_numpy.items()}


In [18]:
def write_onel_module(path, name = "S", variables = None,  integral_expressions = None):
    lines = ["import numpy as np\n",
             "from numba import njit\n",
             "@njit(cache=True, fastmath=True)\n",
              f"def {name}({variables}):\n"]
    
    for rep, value in enumerate(integral_expressions.values()):
        lines.append(f"    if rep == {rep}:\n")
        lines.append(f"        return {printer.doprint(value)}\n")
    lines.append("    raise ValueError(\"Invalid overlap representative\")\n")
    path = path / (name + ".py")
    with open(path, "w", encoding="utf-8") as f:
        f.write("".join(lines))

In [19]:
from pathlib import Path
path = Path.cwd() / "src_live"
write_onel_module(path, name = "S", variables = "rep, Dx, Dy, Dz, P, Q, RAB2, alpha, beta", integral_expressions = integral_expressions)

In [20]:
import timeit
import importlib
from src_live import S
importlib.reload(S)
%timeit S.S(15, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0,1.0)


The slowest run took 7.67 times longer than the fastest. This could mean that an intermediate result is being cached.
536 ns ± 568 ns per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Overlap shell pair

In [58]:
IJKLMN_TO_REP = {(0, 0, 0, 0, 0, 0): (0, (2, 1, 0), True),
 (0, 0, 0, 0, 0, 1): (1, (1, 0, 2), False),
 (0, 0, 0, 0, 1, 0): (1, (2, 0, 1), False),
 (0, 0, 0, 1, 0, 0): (1, (2, 1, 0), False),
 (0, 0, 0, 0, 0, 2): (2, (1, 0, 2), False),
 (0, 0, 0, 0, 1, 1): (3, (0, 2, 1), False),
 (0, 0, 0, 0, 2, 0): (2, (2, 0, 1), False),
 (0, 0, 0, 1, 0, 1): (3, (1, 2, 0), False),
 (0, 0, 0, 1, 1, 0): (3, (2, 1, 0), False),
 (0, 0, 0, 2, 0, 0): (2, (2, 1, 0), False),
 (0, 0, 1, 0, 0, 0): (1, (1, 0, 2), True),
 (0, 1, 0, 0, 0, 0): (1, (2, 0, 1), True),
 (1, 0, 0, 0, 0, 0): (1, (2, 1, 0), True),
 (0, 0, 1, 0, 0, 1): (4, (1, 0, 2), True),
 (0, 0, 1, 0, 1, 0): (6, (0, 2, 1), True),
 (0, 0, 1, 1, 0, 0): (6, (1, 2, 0), True),
 (0, 1, 0, 0, 0, 1): (6, (0, 2, 1), False),
 (0, 1, 0, 0, 1, 0): (4, (2, 0, 1), True),
 (0, 1, 0, 1, 0, 0): (6, (2, 1, 0), True),
 (1, 0, 0, 0, 0, 1): (6, (1, 2, 0), False),
 (1, 0, 0, 0, 1, 0): (6, (2, 1, 0), False),
 (1, 0, 0, 1, 0, 0): (4, (2, 1, 0), True),
 (0, 0, 1, 0, 0, 2): (5, (1, 0, 2), False),
 (0, 0, 1, 0, 1, 1): (7, (0, 1, 2), False),
 (0, 0, 1, 0, 2, 0): (8, (0, 1, 2), False),
 (0, 0, 1, 1, 0, 1): (7, (1, 0, 2), False),
 (0, 0, 1, 1, 1, 0): (9, (1, 0, 2), False),
 (0, 0, 1, 2, 0, 0): (8, (1, 0, 2), False),
 (0, 1, 0, 0, 0, 2): (8, (0, 2, 1), False),
 (0, 1, 0, 0, 1, 1): (7, (0, 2, 1), False),
 (0, 1, 0, 0, 2, 0): (5, (2, 0, 1), False),
 (0, 1, 0, 1, 0, 1): (9, (2, 0, 1), False),
 (0, 1, 0, 1, 1, 0): (7, (2, 0, 1), False),
 (0, 1, 0, 2, 0, 0): (8, (2, 0, 1), False),
 (1, 0, 0, 0, 0, 2): (8, (1, 2, 0), False),
 (1, 0, 0, 0, 1, 1): (9, (2, 1, 0), False),
 (1, 0, 0, 0, 2, 0): (8, (2, 1, 0), False),
 (1, 0, 0, 1, 0, 1): (7, (1, 2, 0), False),
 (1, 0, 0, 1, 1, 0): (7, (2, 1, 0), False),
 (1, 0, 0, 2, 0, 0): (5, (2, 1, 0), False),
 (0, 0, 2, 0, 0, 0): (2, (1, 0, 2), True),
 (0, 1, 1, 0, 0, 0): (3, (0, 2, 1), True),
 (0, 2, 0, 0, 0, 0): (2, (2, 0, 1), True),
 (1, 0, 1, 0, 0, 0): (3, (1, 2, 0), True),
 (1, 1, 0, 0, 0, 0): (3, (2, 1, 0), True),
 (2, 0, 0, 0, 0, 0): (2, (2, 1, 0), True),
 (0, 0, 2, 0, 0, 1): (5, (1, 0, 2), True),
 (0, 0, 2, 0, 1, 0): (8, (0, 2, 1), True),
 (0, 0, 2, 1, 0, 0): (8, (1, 2, 0), True),
 (0, 1, 1, 0, 0, 1): (7, (0, 1, 2), True),
 (0, 1, 1, 0, 1, 0): (7, (0, 2, 1), True),
 (0, 1, 1, 1, 0, 0): (9, (2, 1, 0), True),
 (0, 2, 0, 0, 0, 1): (8, (0, 1, 2), True),
 (0, 2, 0, 0, 1, 0): (5, (2, 0, 1), True),
 (0, 2, 0, 1, 0, 0): (8, (2, 1, 0), True),
 (1, 0, 1, 0, 0, 1): (7, (1, 0, 2), True),
 (1, 0, 1, 0, 1, 0): (9, (2, 0, 1), True),
 (1, 0, 1, 1, 0, 0): (7, (1, 2, 0), True),
 (1, 1, 0, 0, 0, 1): (9, (1, 0, 2), True),
 (1, 1, 0, 0, 1, 0): (7, (2, 0, 1), True),
 (1, 1, 0, 1, 0, 0): (7, (2, 1, 0), True),
 (2, 0, 0, 0, 0, 1): (8, (1, 0, 2), True),
 (2, 0, 0, 0, 1, 0): (8, (2, 0, 1), True),
 (2, 0, 0, 1, 0, 0): (5, (2, 1, 0), True),
 (0, 0, 2, 0, 0, 2): (10, (1, 0, 2), True),
 (0, 0, 2, 0, 1, 1): (11, (0, 1, 2), False),
 (0, 0, 2, 0, 2, 0): (12, (0, 2, 1), True),
 (0, 0, 2, 1, 0, 1): (11, (1, 0, 2), False),
 (0, 0, 2, 1, 1, 0): (13, (1, 0, 2), False),
 (0, 0, 2, 2, 0, 0): (12, (1, 2, 0), True),
 (0, 1, 1, 0, 0, 2): (11, (0, 1, 2), True),
 (0, 1, 1, 0, 1, 1): (14, (0, 2, 1), True),
 (0, 1, 1, 0, 2, 0): (11, (0, 2, 1), True),
 (0, 1, 1, 1, 0, 1): (15, (1, 0, 2), True),
 (0, 1, 1, 1, 1, 0): (15, (2, 0, 1), True),
 (0, 1, 1, 2, 0, 0): (13, (2, 1, 0), True),
 (0, 2, 0, 0, 0, 2): (12, (0, 2, 1), False),
 (0, 2, 0, 0, 1, 1): (11, (0, 2, 1), False),
 (0, 2, 0, 0, 2, 0): (10, (2, 0, 1), True),
 (0, 2, 0, 1, 0, 1): (13, (2, 0, 1), False),
 (0, 2, 0, 1, 1, 0): (11, (2, 0, 1), False),
 (0, 2, 0, 2, 0, 0): (12, (2, 1, 0), True),
 (1, 0, 1, 0, 0, 2): (11, (1, 0, 2), True),
 (1, 0, 1, 0, 1, 1): (15, (1, 0, 2), False),
 (1, 0, 1, 0, 2, 0): (13, (2, 0, 1), True),
 (1, 0, 1, 1, 0, 1): (14, (1, 2, 0), True),
 (1, 0, 1, 1, 1, 0): (15, (2, 1, 0), True),
 (1, 0, 1, 2, 0, 0): (11, (1, 2, 0), True),
 (1, 1, 0, 0, 0, 2): (13, (1, 0, 2), True),
 (1, 1, 0, 0, 1, 1): (15, (2, 0, 1), False),
 (1, 1, 0, 0, 2, 0): (11, (2, 0, 1), True),
 (1, 1, 0, 1, 0, 1): (15, (2, 1, 0), False),
 (1, 1, 0, 1, 1, 0): (14, (2, 1, 0), True),
 (1, 1, 0, 2, 0, 0): (11, (2, 1, 0), True),
 (2, 0, 0, 0, 0, 2): (12, (1, 2, 0), False),
 (2, 0, 0, 0, 1, 1): (13, (2, 1, 0), False),
 (2, 0, 0, 0, 2, 0): (12, (2, 1, 0), False),
 (2, 0, 0, 1, 0, 1): (11, (1, 2, 0), False),
 (2, 0, 0, 1, 1, 0): (11, (2, 1, 0), False),
 (2, 0, 0, 2, 0, 0): (10, (2, 1, 0), True)}

LA_LB_TO_IJKLMN = {(0, 0): [(0, 0, 0, 0, 0, 0)],
 (0, 1): [(0, 0, 0, 0, 0, 1), (0, 0, 0, 0, 1, 0), (0, 0, 0, 1, 0, 0)],
 (0, 2): [(0, 0, 0, 0, 0, 2),
          (0, 0, 0, 0, 1, 1),
          (0, 0, 0, 0, 2, 0),
          (0, 0, 0, 1, 0, 1),
          (0, 0, 0, 1, 1, 0),
          (0, 0, 0, 2, 0, 0)],
 (1, 0): [(0, 0, 1, 0, 0, 0), (0, 1, 0, 0, 0, 0), (1, 0, 0, 0, 0, 0)],
 (1, 1): [(0, 0, 1, 0, 0, 1),
          (0, 0, 1, 0, 1, 0),
          (0, 0, 1, 1, 0, 0),
          (0, 1, 0, 0, 0, 1),
          (0, 1, 0, 0, 1, 0),
          (0, 1, 0, 1, 0, 0),
          (1, 0, 0, 0, 0, 1),
          (1, 0, 0, 0, 1, 0),
          (1, 0, 0, 1, 0, 0)],
 (1, 2): [(0, 0, 1, 0, 0, 2),
          (0, 0, 1, 0, 1, 1),
          (0, 0, 1, 0, 2, 0),
          (0, 0, 1, 1, 0, 1),
          (0, 0, 1, 1, 1, 0),
          (0, 0, 1, 2, 0, 0),
          (0, 1, 0, 0, 0, 2),
          (0, 1, 0, 0, 1, 1),
          (0, 1, 0, 0, 2, 0),
          (0, 1, 0, 1, 0, 1),
          (0, 1, 0, 1, 1, 0),
          (0, 1, 0, 2, 0, 0),
          (1, 0, 0, 0, 0, 2),
          (1, 0, 0, 0, 1, 1),
          (1, 0, 0, 0, 2, 0),
          (1, 0, 0, 1, 0, 1),
          (1, 0, 0, 1, 1, 0),
          (1, 0, 0, 2, 0, 0)],
 (2, 0): [(0, 0, 2, 0, 0, 0),
          (0, 1, 1, 0, 0, 0),
          (0, 2, 0, 0, 0, 0),
          (1, 0, 1, 0, 0, 0),
          (1, 1, 0, 0, 0, 0),
          (2, 0, 0, 0, 0, 0)],
 (2, 1): [(0, 0, 2, 0, 0, 1),
          (0, 0, 2, 0, 1, 0),
          (0, 0, 2, 1, 0, 0),
          (0, 1, 1, 0, 0, 1),
          (0, 1, 1, 0, 1, 0),
          (0, 1, 1, 1, 0, 0),
          (0, 2, 0, 0, 0, 1),
          (0, 2, 0, 0, 1, 0),
          (0, 2, 0, 1, 0, 0),
          (1, 0, 1, 0, 0, 1),
          (1, 0, 1, 0, 1, 0),
          (1, 0, 1, 1, 0, 0),
          (1, 1, 0, 0, 0, 1),
          (1, 1, 0, 0, 1, 0),
          (1, 1, 0, 1, 0, 0),
          (2, 0, 0, 0, 0, 1),
          (2, 0, 0, 0, 1, 0),
          (2, 0, 0, 1, 0, 0)],
 (2, 2): [(0, 0, 2, 0, 0, 2),
          (0, 0, 2, 0, 1, 1),
          (0, 0, 2, 0, 2, 0),
          (0, 0, 2, 1, 0, 1),
          (0, 0, 2, 1, 1, 0),
          (0, 0, 2, 2, 0, 0),
          (0, 1, 1, 0, 0, 2),
          (0, 1, 1, 0, 1, 1),
          (0, 1, 1, 0, 2, 0),
          (0, 1, 1, 1, 0, 1),
          (0, 1, 1, 1, 1, 0),
          (0, 1, 1, 2, 0, 0),
          (0, 2, 0, 0, 0, 2),
          (0, 2, 0, 0, 1, 1),
          (0, 2, 0, 0, 2, 0),
          (0, 2, 0, 1, 0, 1),
          (0, 2, 0, 1, 1, 0),
          (0, 2, 0, 2, 0, 0),
          (1, 0, 1, 0, 0, 2),
          (1, 0, 1, 0, 1, 1),
          (1, 0, 1, 0, 2, 0),
          (1, 0, 1, 1, 0, 1),
          (1, 0, 1, 1, 1, 0),
          (1, 0, 1, 2, 0, 0),
          (1, 1, 0, 0, 0, 2),
          (1, 1, 0, 0, 1, 1),
          (1, 1, 0, 0, 2, 0),
          (1, 1, 0, 1, 0, 1),
          (1, 1, 0, 1, 1, 0),
          (1, 1, 0, 2, 0, 0),
          (2, 0, 0, 0, 0, 2),
          (2, 0, 0, 0, 1, 1),
          (2, 0, 0, 0, 2, 0),
          (2, 0, 0, 1, 0, 1),
          (2, 0, 0, 1, 1, 0),
          (2, 0, 0, 2, 0, 0)]}

In [59]:
def overlap_shell_pair(exp_a, coeff_a, norm_a, center_a,
                       exp_b, coeff_b, norm_b, center_b,
                       ijklmn):
    dim_a = norm_a.shape[1]
    dim_b = norm_b.shape[1]
    out = np.zeros((dim_a, dim_b), dtype = np.float64)
    RAB2 = (center_a[0]-center_b[0])**2 + (center_a[1]-center_b[1])**2 + (center_a[2]-center_b[2])**2

    for idx in range(len(ijklmn)):
        ia = idx // dim_b
        ib = idx % dim_b
        ijklmn_tuple = ijklmn[idx]
        rep, perm, swap = IJKLMN_TO_REP[ijklmn_tuple]
        if swap == False:
            A,B = center_a, center_b
        else:
            A,B = center_b, center_a
        Dx = A[perm[0]] - B[perm[0]]
        Dy = A[perm[1]] - B[perm[1]]
        Dz = A[perm[2]] - B[perm[2]]
        val = 0.0
        for pa in range(exp_a.shape[0]):
            alpha_a = exp_a[pa]
            ca = coeff_a[pa] * norm_a[pa, ia]
            for pb in range(exp_b.shape[0]):
                alpha_b = exp_b[pb]
                cb = coeff_b[pb] * norm_b[pb, ib]
                P = alpha_a + alpha_b
                Q = alpha_a * alpha_b
                alpha = alpha_b if swap else alpha_a
                beta = alpha_a if swap else alpha_b
                val += ca * cb * S.S(rep, Dx, Dy, Dz, P, Q, RAB2, alpha, beta)

        out[ia, ib] = val
    return out
    

In [60]:
%pip install -e .

Obtaining file:///Users/rolandmitric/MASTER_PROGRAMMING_2026/live_notebooks
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for theochem_live (pyproject.toml) ... done
  Created wheel for theochem_live: filename=theochem_live-0.1.0-0.editable-py3-none-any.whl size=2776 sha256=ce8a612c93165f44062a3644f6c2fe6643db0a4097db735ca3e64dc0dd4c3eb6
  Stored in directory: /private/var/folders/cp/hbk0qs4x4ggb5z_vv2hkmcgw0000gn/T/pip-ephem-wheel-cache-rr0mbvhw/wheels/a8/61/bd/318087ecc54d7e0ca7146aa39c26758304348afc27f0ad3eb7
Successfully built theochem_live
  Attempting uninstall: theochem_live
    Found existing installation: theochem_live 0.1.0
    Uninstalling theochem_live-0.1.0:
      Successfully uninstalled theochem_live-0.1.0

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: 

## Molecular test data

In [61]:
from src_live import Molecule, BasisSet
benzene_xyz = """12
Benzene molecule
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000
"""
benzene = Molecule.from_string(benzene_xyz)
print(benzene.to_string())
sto3g = BasisSet("sto-3g")
sto3g.download_from_bse(["H", "C", "O", "N", "B"])
print(sto3g)

12
Generated by Molecule Class
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000
Dump data to sto-3g.json
Basis set: sto-3g
Element: H
l = 0, exponents = [3.42525091 0.62391373 0.1688554 ], coefficients = [0.15432897 0.53532814 0.44463454]
Element: B
l = 0, exponents = [48.79111318  8.88736217  2.40526704], coefficients = [0.15432897 0.53532814 0.44463454]
l = 0, exponents = [2.23695614 0.5198205  0.16906176], coefficients = [-0.09996723  0.39951283  0.70011547]
l = 1, exponents = [2.23695614 0.5198205  0.16906176], coefficients = [0.15591627 0.60768372 0.39195739]
Element: C
l = 0, exponents = [71.61683735 13.04509632  3.53051216], coefficients = [0.15432897 0.53532814 0.44463454]
l

## Molecule integrals class

In [83]:
from dataclasses import dataclass, field
import copy
import numpy as np
from numba import njit
from numba.typed import List

ANG2BOHR = 1.8897261254578281

@dataclass
class MolecularIntegrals:
    molecule: Molecule
    basis_set: BasisSet
    #stuff that we don't initialize
    shells: List[Shell] = field(default_factory=List, init=False)
    exp_data: List = field(init = False)
    coeff_data: List = field(init = False)
    norm_data: List = field(init = False)
    center_data: List = field(init = False)
    ijklmn_data = List() 

    def __post_init__(self):
        self.shells = []
        for atom in self.molecule.atoms:
            atom_basis = self.basis_set.elements[atom.symbol]
            for shell in atom_basis:
                shell_copy = copy.deepcopy(shell)
                shell_copy.set_center(np.asarray(atom.coord, dtype=np.float64) * ANG2BOHR)
                self.shells.append(shell_copy)

        self.exp_data = List()
        self.coeff_data = List()
        self.norm_data = List()
        self.center_data = List()
        for shell in self.shells:
            self.exp_data.append(np.ascontiguousarray(shell.exponents, dtype = np.float64))
            self.coeff_data.append(np.ascontiguousarray(shell.coefficients, dtype = np.float64))
            self.norm_data.append(np.ascontiguousarray(shell.norm_factors, dtype = np.float64))
            self.center_data.append(np.ascontiguousarray(shell.center, dtype = np.float64))
        
        # create pair data for one electron integrals
        self.get_pair_data()

    def get_pair_data(self):
        pair_a_list = List()
        pair_b_list = List()
        for i, shell_a in enumerate(self.shells):
            for j, shell_b in enumerate(self.shells):
                pair_a_list.append(i)
                pair_b_list.append(j)
                ijk_lmn = LA_LB_TO_IJKLMN[(shell_a.l, shell_b.l)]
                self.ijklmn_data.append(ijk_lmn)
        self.pair_a = np.asarray(pair_a_list, dtype = np.int64) 
        self.pair_b = np.asarray(pair_b_list, dtype = np.int64)  

    def one_electron_driver(self, engine = overlap_shell_pair):
        n_pairs = len(self.pair_a)
        out = List()
        for idx in range(n_pairs):
            i = self.pair_a[idx]
            j = self.pair_b[idx]
            shell_a = self.shells[i]
            shell_b = self.shells[j]
            exp_a = self.exp_data[i]
            coeff_a = self.coeff_data[i]
            norm_a = self.norm_data[i]
            center_a = self.center_data[i]
            exp_b = self.exp_data[j]
            coeff_b = self.coeff_data[j]
            norm_b = self.norm_data[j]
            center_b = self.center_data[j]
            ijklmn = self.ijklmn_data[idx]
            out.append(engine(exp_a, coeff_a, norm_a, center_a,
                                         exp_b, coeff_b, norm_b, center_b,
                                         ijklmn))
        return out

In [84]:
molints = MolecularIntegrals(benzene, sto3g)
molints.one_electron_driver()


ListType[array(float64, 2d, C)]([[[1.]], [[0.2483624]], [[0. 0. 0.]], [[8.12513382e-07]], [[0.03726619]], [[ 0.          0.03089579 -0.05351303]], [[1.36640992e-17]], [[0.00175427]], [[ 0.          0.00351662 -0.00203032]], [[5.60740989e-23]], [[0.00039553]], [[0.        0.0010552 0.       ]], [[1.36640992e-17]], [[0.00175427]], [[0.         0.00351662 0.00203032]], [[8.12513382e-07]], [[0.03726619]], [[0.         0.03089579 0.05351303]], [[0.0626298]], [[0.00533126]], [[9.12226733e-05]], [[1.19794524e-05]], [[9.12226733e-05]], [[0.00533126]], [[0.2483624]], [[1.]], [[0. 0. 0.]], [[0.03726619]], [[0.36328505]], [[ 0.          0.19345341 -0.33507081]], [[0.00175427]], [[0.05955609]], [[ 0.          0.07612928 -0.04395322]], [[0.00039553]], [[0.02542269]], [[0.        0.0409766 0.       ]], [[0.00175427]], [[0.05955609]], [[0.         0.07612928 0.04395322]], [[0.03726619]], [[0.36328505]], [[0.         0.19345341 0.33507081]], [[0.49215329]], [[0.09611724]], [[0.00641537]], [[0.00182869